<a href="https://colab.research.google.com/github/sksundaram-learning/my-repo/blob/master/RFP_Requirements_Responses.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install outlines
!pip install lancedb
!pip install openai
!pip install pypdf2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.4/98.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.2/39.2 MB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.8/256.8 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.3 MB/s eta 0:00:00


# Processed of RFP PDF Documents Locally on the Laptop using the following code

In [ ]:
import outlines
from openai import OpenAI

# Create an OpenAI client instance
openai_client = OpenAI()

# The outlines package may not provide a `from_openai` helper.
# Use it if available; otherwise fall back to using the OpenAI client directly.
if hasattr(outlines, "from_openai"):
	model = outlines.from_openai(openai_client, "gpt-4o")
else:
	# outlines.from_openai is not present in this version of outlines.
	# Use the OpenAI client directly as the model object for downstream calls.
	# Replace this fallback with the proper outlines integration when available.
	model = openai_client

prompt = "Write a short story about a cat."

# If outlines returned a callable wrapper, call it directly.
if callable(model):
	result = model(prompt)
	# Try to print a sensible text representation
	if hasattr(result, "text"):
		print(result.text)
	else:
		print(result)

else:
	# model is an OpenAI client instance (not callable). Use the client APIs.
	# Try the Responses API first, then fall back to chat completions.
	try:
		resp = model.responses.create(model="gpt-4o", input=prompt)
		# Prefer convenient properties if available
		if hasattr(resp, "output_text"):
			print(resp.output_text)
		else:
			# Try to extract nested text if present (common Responses structure)
			try:
				output = getattr(resp, "output", None)
				if output and len(output) > 0:
					content = getattr(output[0], "content", None)
					if content and len(content) > 0 and hasattr(content[0], "get"):
						# content items may be dict-like
						print(content[0].get("text") or content[0].get("content") or content[0])
						raise SystemExit
					elif content and len(content) > 0 and hasattr(content[0], "text"):
						print(content[0].text)
						raise SystemExit
				# Fallback to printing full response object
				print(resp)
			except SystemExit:
				# printed successfully inside the nested extraction
				pass
	except Exception:
		# Fallback to chat completions if Responses API call fails
		try:
			resp = model.chat.completions.create(model="gpt-4o", messages=[{"role": "user", "content": prompt}])
			choices = getattr(resp, "choices", None)
			if choices and len(choices) > 0:
				choice = choices[0]
				# choice.message may be dict-like or object
				msg = getattr(choice, "message", None) or (choice.get("message") if hasattr(choice, "get") else None)
				if isinstance(msg, dict):
					print(msg.get("content"))
				elif hasattr(msg, "get"):
					print(msg.get("content"))
				else:
					# try typical attribute
					print(getattr(choice, "message", choice))
			else:
				print(resp)
		except Exception as e:
			# Report error if both attempts fail
			print("Failed to call model with client APIs:", e)

ModuleNotFoundError: No module named 'outlines'

### Extract RFP Requirements and Responses from local PDF files and create a structured lancedb database

In [ ]:
import os
import json
import glob
from pathlib import Path
from PyPDF2 import PdfReader
import lancedb
import outlines
from openai import OpenAI
from typing import Optional, List, Dict

def process_pdfs_to_lancedb(
    folder_path: str = "F:\\SNG-Arun\\SNG-Core-PDF-Docs",
    db_path: str = "/content/pdf_data.db",
    table_name: str = "pdf_extractions"
) -> None:
    """
    Process all PDFs recursively from a folder, extract requirements using GPT-4o,
    and store the results in a LanceDB table.

    Args:
        folder_path: Root folder path to recursively search for PDFs
        db_path: Path to store LanceDB database
        table_name: Name of the LanceDB table to store results
    """

    # Initialize OpenAI client
    openai_client = OpenAI()

    # Initialize model
    if hasattr(outlines, "from_openai"):
        model = outlines.from_openai(openai_client, "gpt-4o")
    else:
        model = openai_client

    # Initialize LanceDB
    db = lancedb.connect(db_path)

    # Find all PDF files recursively
    folder_path = Path(folder_path)
    pdf_files = sorted(folder_path.rglob("*.pdf"))

    if not pdf_files:
        print(f"No PDF files found in {folder_path}")
        return

    print(f"Found {len(pdf_files)} PDF files. Processing...")

    all_results = []

    for idx, pdf_path in enumerate(pdf_files, 1):
        try:
            print(f"\n[{idx}/{len(pdf_files)}] Processing: {pdf_path.name}")

            # Extract text from PDF
            reader = PdfReader(pdf_path)
            pages = [p.extract_text() or "" for p in reader.pages]
            document_text = "\n\n".join(
                f"---PAGE {i+1}---\n{txt}" for i, txt in enumerate(pages)
            )

            # Create extraction prompt
            prompt = f"""
You are given an RFP document. Extract a hierarchical representation of sections and requirements, and for each requirement provide a concise suggested response. Return ONLY valid JSON structured as:

{{
  "source_file": "<filename>",
  "total_pages": <page count>,
  "sections": [
    {{
      "title": "<section title or best guess>",
      "start_page": <page number>,
      "requirements": [
        {{
          "id": "<requirement id if present or generated>",
          "text": "<requirement text>",
          "page": <page number>,
          "subrequirements": [ ... nested same schema ... ],
          "recommended_response": "<concise response, max ~200 words>"
        }}
      ]
    }}
  ]
}}

Preserve hierarchy, include page numbers, and keep responses concise. Now process the document content below.

Document:
{document_text}
"""

            # Call the model
            result_text = _call_model(model, prompt)

            # Parse JSON from result
            try:
                # Try to extract JSON from markdown code blocks if present
                if "```json" in result_text:
                    json_str = result_text.split("```json")[1].split("```")[0].strip()
                elif "```" in result_text:
                    json_str = result_text.split("```")[1].split("```")[0].strip()
                else:
                    json_str = result_text

                extracted_data = json.loads(json_str)
            except json.JSONDecodeError as e:
                print(f"  ⚠️  Failed to parse JSON: {e}")
                extracted_data = {
                    "source_file": pdf_path.name,
                    "total_pages": len(pages),
                    "error": "Failed to parse response as JSON",
                    "raw_response": result_text[:500]  # Store first 500 chars
                }

            # Add metadata
            extracted_data["file_path"] = str(pdf_path)
            extracted_data["file_name"] = pdf_path.name
            extracted_data["total_pages"] = len(pages)

            all_results.append(extracted_data)
            print(f"  ✓ Successfully processed")

        except Exception as e:
            print(f"  ✗ Error processing {pdf_path.name}: {e}")
            all_results.append({
                "file_path": str(pdf_path),
                "file_name": pdf_path.name,
                "error": str(e)
            })

    # Store results in LanceDB
    if all_results:
        print(f"\nStoring {len(all_results)} results in LanceDB...")
        try:
            # Create or overwrite table
            db.create_table(table_name, data=all_results, mode="overwrite")
            print(f"✓ Successfully stored results in table '{table_name}'")
            print(f"  Database location: {db_path}")
            print(f"  Total records: {len(all_results)}")
        except Exception as e:
            print(f"✗ Error storing in LanceDB: {e}")


def _call_model(model, prompt: str) -> str:
    """Helper function to call the model with fallback strategies."""

    try:
        if callable(model):
            result = model(prompt)
            return getattr(result, "text", str(result))
        else:
            # Try responses API
            try:
                resp = model.responses.create(model="gpt-4o", input=prompt)
                if hasattr(resp, "output_text"):
                    return resp.output_text
                output = getattr(resp, "output", None)
                if output and len(output) > 0:
                    content = getattr(output[0], "content", None)
                    if content and len(content) > 0:
                        return content[0].get("text") if hasattr(content[0], "get") else getattr(content[0], "text", str(content[0]))
            except:
                pass

            # Fallback to chat completions
            resp = model.chat.completions.create(
                model="gpt-4o",
                messages=[{"role": "user", "content": prompt}]
            )
            choices = getattr(resp, "choices", [])
            if choices:
                msg = getattr(choices[0], "message", None)
                if msg:
                    return getattr(msg, "content", str(msg))
            return str(resp)

    except Exception as e:
        raise Exception(f"Failed to call model: {e}")


# Usage example
if __name__ == "__main__":
    process_pdfs_to_lancedb(
        folder_path="F:\\SNG-Arun\\SNG-Core-PDF-Docs",
        db_path="./pdf_data.db",
        table_name="pdf_extractions"
    )

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

## Correct the incorrect 'file_name' that appears in the dataframe 'source_file' column

In [ ]:
import pandas as pd
import lancedb
import json

# Connect to the database
db = lancedb.connect("/content/pdf_data.db")
table = db.open_table("pdf_extractions")

# Get all records as a list of dicts
records = table.search().to_list()

# Create a dataframe
df = pd.DataFrame(records)

df['source_file'] = df.apply(
    lambda row: row['file_name'] if row['source_file'] == '<filename>' else row['source_file'],
    axis=1
)

# Display basic info
print(f"Total records: {len(df)}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nDataFrame shape: {df.shape}")
print("\n" + "="*80)

# Display summary - only show columns that exist
summary_cols = [col for col in ['file_name', 'total_pages', 'error'] if col in df.columns]
print(df[summary_cols].to_string())

# Display full dataframe with all columns
print("\n" + "="*80)
print("Full DataFrame:")
print(df.to_string())

# Show data types
print("\n" + "="*80)
print("\nData types:")
print(df.dtypes)

# Display the dataframe for interactive viewing
df

Total records: 9

Columns: ['source_file', 'total_pages', 'sections', 'file_path', 'file_name']

DataFrame shape: (9, 5)

                                                                  file_name  total_pages
0               SNG PROJECT EXPERIENCE MATRIX SHEET_City of Manteca RFP.pdf            1
1  SNG PROJECT EXPERIENCE MATRIX SHEET_City of Martinez-final draft (1).pdf            1
2                       11302023 proposal for City of Mountain view (1).pdf           48
3            STATEMENT OF QUALS FOR ON CALL PROFESSIONAL SERVICES LARPD.pdf           36
4                                                            Louis Mata.pdf            1
5                                        MOUNTAIN VIEW All staff resume.pdf           19
6                                                                   NG1.pdf            1
7                                              Addendum No1  RFQ 251084.pdf            3
8                                 RFQ Construction CM_MVF00-51 - NA (1).pdf  

,source_file,total_pages,sections,file_path,file_name
0,RFP_Scope_of_Work_Document,1,"[{'requirements': [{'id': '1.1', 'page': 1, 'r...",F:\SNG-Arun\SNG-Core-PDF-Docs\Other Docs\SNG P...,SNG PROJECT EXPERIENCE MATRIX SHEET_City of Ma...
1,SNG PROJECT EXPERIENCE MATRIX SHEET_City of Ma...,1,None,F:\SNG-Arun\SNG-Core-PDF-Docs\Other Docs\SNG P...,SNG PROJECT EXPERIENCE MATRIX SHEET_City of Ma...
2,11302023 proposal for City of Mountain view (1...,48,None,F:\SNG-Arun\SNG-Core-PDF-Docs\Proposals\113020...,11302023 proposal for City of Mountain view (1...
3,STATEMENT OF QUALS FOR ON CALL PROFESSIONAL SE...,36,"[{'requirements': [{'id': 'req-001', 'page': 2...",F:\SNG-Arun\SNG-Core-PDF-Docs\Proposals\STATEM...,STATEMENT OF QUALS FOR ON CALL PROFESSIONAL SE...
4,Louis Mata.pdf,1,None,F:\SNG-Arun\SNG-Core-PDF-Docs\Resume Format\Lo...,Louis Mata.pdf
5,MOUNTAIN VIEW All staff resume.pdf,19,None,F:\SNG-Arun\SNG-Core-PDF-Docs\Resume Format\MO...,MOUNTAIN VIEW All staff resume.pdf
6,NG1.pdf,1,None,F:\SNG-Arun\SNG-Core-PDF-Docs\Resume Format\NG...,NG1.pdf
7,Addendum No1 RFQ 251084.pdf,3,"[{'requirements': [{'id': 'R1', 'page': 1, 're...",F:\SNG-Arun\SNG-Core-PDF-Docs\RFP Docs\Addendu...,Addendum No1 RFQ 251084.pdf
8,RFQ Construction CM_MVF00-51 - NA (1).pdf,27,None,F:\SNG-Arun\SNG-Core-PDF-Docs\RFP Docs\RFQ Con...,RFQ Construction CM_MVF00-51 - NA (1).pdf


## Flatten the nested JSON structure for requirements analysis

In [ ]:
import pandas as pd

# Flatten the requirements from sections
flattened_data = []

for idx, row in df.iterrows():
    source_file = row['file_name']
    file_path = row['file_path']
    total_pages = row['total_pages']
    sections = row['sections']

    # Skip if sections is None or empty
    if not sections:
        continue

    # Iterate through each section
    for section in sections:
        section_title = section.get('title', 'N/A')
        section_start_page = section.get('start_page', 'N/A')

        # Get requirements from the section
        requirements = section.get('requirements', [])

        for req in requirements:
            flattened_data.append({
                'file_name': source_file,
                'file_path': file_path,
                'total_pages': total_pages,
                'section_title': section_title,
                'section_start_page': section_start_page,
                'requirement_id': req.get('id', 'N/A'),
                'requirement_text': req.get('text', 'N/A'),
                'requirement_page': req.get('page', 'N/A'),
                'recommended_response': req.get('recommended_response', 'N/A'),
            })

# Create flattened dataframe
df_flattened = pd.DataFrame(flattened_data)

print(f"Total requirements extracted: {len(df_flattened)}")
print(f"\nFlattened DataFrame columns: {list(df_flattened.columns)}")
print(f"\nFlattened DataFrame shape: {df_flattened.shape}")
print("\n" + "="*100)
print(df_flattened[['file_name', 'section_title', 'requirement_id', 'requirement_text']].head(20).to_string())
print("\n" + "="*100)
print("\nFull Flattened DataFrame:")
df_flattened

Total requirements extracted: 27

Flattened DataFrame columns: ['file_name', 'file_path', 'total_pages', 'section_title', 'section_start_page', 'requirement_id', 'requirement_text', 'requirement_page', 'recommended_response']

Flattened DataFrame shape: (27, 9)

                                                         file_name                            section_title requirement_id                                                                                                                                                                                                                           requirement_text
0      SNG PROJECT EXPERIENCE MATRIX SHEET_City of Manteca RFP.pdf          Administrative/General Services            1.1                                                                                                                                                                         Attend review and coordination meetings with City staff as needed.
1      SNG PROJECT EX

,file_name,file_path,total_pages,section_title,section_start_page,requirement_id,requirement_text,requirement_page,recommended_response
0,SNG PROJECT EXPERIENCE MATRIX SHEET_City of Ma...,F:\SNG-Arun\SNG-Core-PDF-Docs\Other Docs\SNG P...,1,Administrative/General Services,1,1.1,Attend review and coordination meetings with C...,1,We will establish a flexible schedule to ensur...
1,SNG PROJECT EXPERIENCE MATRIX SHEET_City of Ma...,F:\SNG-Arun\SNG-Core-PDF-Docs\Other Docs\SNG P...,1,Administrative/General Services,1,1.2,Discuss plan review comments with City staff o...,1,Our team will prepare to discuss and provide t...
2,SNG PROJECT EXPERIENCE MATRIX SHEET_City of Ma...,F:\SNG-Arun\SNG-Core-PDF-Docs\Other Docs\SNG P...,1,Administrative/General Services,1,1.3,Provide initial and subsequent plan checks for...,1,We commit to delivering initial and follow-up ...
3,SNG PROJECT EXPERIENCE MATRIX SHEET_City of Ma...,F:\SNG-Arun\SNG-Core-PDF-Docs\Other Docs\SNG P...,1,Administrative/General Services,1,1.4,"Submit plan review comments, revisions, edits,...",1,We will utilize a secure electronic platform t...
4,SNG PROJECT EXPERIENCE MATRIX SHEET_City of Ma...,F:\SNG-Arun\SNG-Core-PDF-Docs\Other Docs\SNG P...,1,Administrative/General Services,1,1.5,Complete each review within agreed timeframes ...,1,All reviews will be completed within the desig...
5,SNG PROJECT EXPERIENCE MATRIX SHEET_City of Ma...,F:\SNG-Arun\SNG-Core-PDF-Docs\Other Docs\SNG P...,1,Administrative/General Services,1,1.6,Review rough and fine grading plans.,1,We will conduct detailed reviews of grading pl...
6,SNG PROJECT EXPERIENCE MATRIX SHEET_City of Ma...,F:\SNG-Arun\SNG-Core-PDF-Docs\Other Docs\SNG P...,1,Development Services,1,2.1,"Review civil plans and notes for sewer, water,...",1,Our engineers have expertise in checking for c...
7,SNG PROJECT EXPERIENCE MATRIX SHEET_City of Ma...,F:\SNG-Arun\SNG-Core-PDF-Docs\Other Docs\SNG P...,1,Development Services,1,2.2,Review erosion and sediment control plans.,1,We conduct comprehensive reviews of erosion an...
8,SNG PROJECT EXPERIENCE MATRIX SHEET_City of Ma...,F:\SNG-Arun\SNG-Core-PDF-Docs\Other Docs\SNG P...,1,Development Services,1,2.3,Verify floodplain status and ensure compliance...,1,Our team will rigorously verify floodplain sta...
9,SNG PROJECT EXPERIENCE MATRIX SHEET_City of Ma...,F:\SNG-Arun\SNG-Core-PDF-Docs\Other Docs\SNG P...,1,Development Services,1,2.4,"Review technical reports, including geotechnic...",1,We will review all technical reports meticulou...
